[Course material - Sustain.Brussels - "Avdanced AI workflows with LLM" - 20/04/2026 - 22/04/2026](https://github.com/Yannael/gen-ai-sustain-brussels-2604).

# Thinking Mode in Qwen3.5-0.8B

Modern instruct models can do more than just answer — they can *think out loud* before replying.

**Qwen3.5-0.8B** is a small but capable model from the Qwen3.5 series that supports a **thinking mode**: when enabled, the model first produces an explicit chain-of-thought wrapped in `<think>…</think>` tags, then outputs its final answer. This mirrors how larger reasoning models (like DeepSeek v3, Gemini 3.1 or GPT 5.4) work.

## What you will learn

| Section | What it shows |
|---------|---------------|
| §4 | Load Qwen3.5-0.8B on GPU with `transformers` |
| §5 | Generate a response **without** thinking mode |
| §6 | Generate a response **with** thinking mode |
| §7 | Parse and display the chain-of-thought separately |

## How thinking mode works

The switch is a parameter passed to `tokenizer.apply_chat_template()`:

```python
# OFF — model answers directly
tokenizer.apply_chat_template(messages, enable_thinking=False, ...)

# ON — model first reasons inside <think>…</think>, then answers
tokenizer.apply_chat_template(messages, enable_thinking=True, ...)
```

> **Note:** Qwen3.5 does **not** support the Qwen3 soft-switch tokens (`/think`, `/no_think` appended to the user message). Use the `enable_thinking` parameter instead.

> **Warning:** The 0.8B model is prone to *thinking loops* — it may keep reasoning without stopping. Always set a reasonable `max_new_tokens` limit and use the recommended sampling parameters.

## 1. Install dependencies

In [ ]:
!pip install --quiet transformers accelerate torch openai
!pip install --quiet -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 95.0 MB/s eta 0:00:00


## 2. Imports

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import re

## 3. Load Qwen3.5-0.8B on GPU

We load the model in **bfloat16** (half-precision) to reduce memory usage while preserving numerical stability. `device_map="cuda"` places the whole model on the GPU.

In [ ]:
MODEL_NAME = "Qwen/Qwen3.5-0.8B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
)
model.eval()

print(f"Model:      {MODEL_NAME}")
print(f"Device:     {next(model.parameters()).device}")
print(f"Dtype:      {next(model.parameters()).dtype}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:85: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/1.75G [00:00<?, ?B/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Model:      Qwen/Qwen3.5-0.8B
Device:     cuda:0
Dtype:      torch.bfloat16
Parameters: 752M


## 4. Generation WITHOUT thinking mode

With `enable_thinking=False`, the model skips the chain-of-thought entirely and answers directly — just like a regular instruct model.



In [ ]:
messages = [{"role": "system", "content": "You are a helpful assistant."}, {"role": "user", "content": "What is 2+2?"}]

# Apply chat template with thinking mode OFF
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

print("=== Rendered prompt (no thinking) ===")
print(repr(text))

=== Rendered prompt (no thinking) ===
'<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\nWhat is 2+2?<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'


In [ ]:
inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=1.0,
        do_sample=True,
    )

# Only decode the newly generated tokens (skip the prompt)
new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
response = tokenizer.decode(new_tokens, skip_special_tokens=True)

print("=== Response WITHOUT thinking mode ===")
print(response)

Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


=== Response WITHOUT thinking mode ===
The sum of 2 and 2 is 4. This is equal to the number of the digits **4**.



The answer is short and direct — no reasoning steps are visible.

## 5. Generation WITH thinking mode

With `enable_thinking=True`, the model first emits a chain-of-thought block enclosed in `<think>…</think>` special tokens, then produces its final answer.

We use `skip_special_tokens=False` so the `<think>` tags are visible in the raw output.

In [ ]:
# Apply chat template with thinking mode ON
text_think = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,
)

print("=== Rendered prompt (with thinking) ===")
print(repr(text_think))

=== Rendered prompt (with thinking) ===
'<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\nWhat is 2+2?<|im_end|>\n<|im_start|>assistant\n<think>\n'


In [ ]:
inputs_think = tokenizer(text_think, return_tensors="pt").to(model.device)

with torch.no_grad():
    output_ids_think = model.generate(
        **inputs_think,
        max_new_tokens=2048,   # cap to avoid thinking loops
        temperature=0.1,
        do_sample=True,
    )

new_tokens_think = output_ids_think[0][inputs_think["input_ids"].shape[1]:]
# Keep special tokens so we can see <think> and </think>
full_response = tokenizer.decode(new_tokens_think, skip_special_tokens=False)

print("=== Response WITH thinking mode (raw) ===")
print(full_response)

Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


=== Response WITH thinking mode (raw) ===
Thinking Process:

1.  **Analyze the Request:** The user is asking a simple arithmetic question: "What is 2+2?"

2.  **Identify the Core Task:** Calculate the sum of 2 and 2.

3.  **Determine the Output:** The result is 4.

4.  **Format the Output:** Keep it concise and direct.

5.  **Review Constraints:** The user didn't specify a format (like a math problem, a joke, etc.), so a standard answer is best.

6.  **Final Decision:** "4" or "2 + 2 = 4". I'll provide the answer clearly.

7.  **Safety Check:** No sensitive content, no harmful instructions. Just a math question.

8.  **Construct Response:** "2 + 2 = 4".
</think>

2 + 2 = 4<|im_end|>
<|endoftext|>


In [ ]:
full_response = repr(text_think)+ full_response

## 7. Comparing the two modes

| | Without thinking | With thinking |
|---|---|---|
| **Output tokens** | Few | Many (reasoning + answer) |
| **Latency** | Fast | Slower |
| **Visible reasoning** | No | Yes (`<think>…</think>`) |
| **Best for** | Simple Q&A, chat | Math, logic, multi-step problems |
| **Risk** | — | Thinking loops on small models |

For a simple question like `2+2`, thinking mode adds overhead with little benefit. It becomes valuable for complex multi-step reasoning tasks.